# Жизненный цикл проверки и техническая модель

Эта вычислительная тетрадь показывает, как фонд оценочных средств переносится в техническую модель платформы.

## 1. Загрузка данных

In [ ]:
from pathlib import Path
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

base = Path.cwd()
if not (base / 'data' / 'structured_data.yaml').exists():
    candidates = list(Path.cwd().glob('**/branch_05_fos/data/structured_data.yaml'))
    if candidates:
        base = candidates[0].parents[1]

with open(base / 'data' / 'structured_data.yaml', encoding='utf-8') as f:
    data = yaml.safe_load(f)

(base / 'assets' / 'diagrams').mkdir(parents=True, exist_ok=True)
print(data['branch']['title'])

## 2. Жизненный цикл отправки работы

Работа слушателя не перезаписывается при доработке. Каждая новая версия фиксируется как отдельная попытка отправки.

In [ ]:
transitions = data['status_model']['transitions']
G = nx.DiGraph()
for transition in transitions:
    G.add_edge(transition['from'], transition['to'], label=transition['event'])

fig, ax = plt.subplots(figsize=(11, 6))
pos = nx.spring_layout(G, seed=11)
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2200)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=9)
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle='-|>', arrowsize=18)
edge_labels = {(u, v): d['label'] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7, ax=ax)
ax.set_title('Жизненный цикл отправки работы')
ax.axis('off')
plt.tight_layout()
path = base / 'assets' / 'diagrams' / 'submission_lifecycle.png'
plt.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
print(path)

## 3. Сущности технической модели

Таблица показывает соответствие русских методических понятий и технических идентификаторов платформы.

In [ ]:
entities = pd.DataFrame(data['platform_entities'])
entities

## 4. Граф технических связей оценивания

Граф показывает минимальный путь от контрольного элемента до итогового документа об обучении.

In [ ]:
edges = [
    ('AssessmentItem', 'AssignmentSubmission'),
    ('GradeScheme', 'AssessmentItem'),
    ('AssignmentSubmission', 'SubmissionAttempt'),
    ('SubmissionAttempt', 'AssignmentReview'),
    ('AssignmentReview', 'Grade'),
    ('Grade', 'CompetencyStatus'),
    ('CompetencyStatus', 'CourseFinalStatus'),
    ('CourseFinalStatus', 'IssuedCertificate'),
]
G = nx.DiGraph()
G.add_edges_from(edges)

fig, ax = plt.subplots(figsize=(13, 6))
pos = nx.spring_layout(G, seed=5)
nx.draw_networkx(G, pos=pos, ax=ax, with_labels=True, node_size=2300, font_size=9, arrows=True, arrowstyle='-|>', arrowsize=18)
ax.set_title('Техническая модель оценивания и выдачи итогового статуса')
ax.axis('off')
plt.tight_layout()
path = base / 'assets' / 'diagrams' / 'assessment_entities_graph.png'
plt.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
print(path)

## 5. Демонстрация правила освоения компетенции

Компетенция считается освоенной, если принят артефакт и пройдена тестовая или практическая проверка.

In [ ]:
demo = pd.DataFrame([
    {'competency':'ПК-ИИ-1','artifact_status':'accepted','test_status':'pass'},
    {'competency':'ПК-ИИ-2','artifact_status':'accepted','test_status':'pass'},
    {'competency':'ПК-ИИ-3','artifact_status':'needs_revision','test_status':'pass'},
    {'competency':'ПК-ИИ-4','artifact_status':'accepted','test_status':'fail'},
])

def competency_result(row):
    return 'освоена' if row['artifact_status'] == 'accepted' and row['test_status'] == 'pass' else 'не освоена'

demo['competency_result'] = demo.apply(competency_result, axis=1)
demo

## 6. Вывод

Техническая модель должна хранить попытки, проверки и итоговые решения отдельно. Это позволяет подтвердить компетенции, выполнить доработку без потери истории и подготовить данные для документа об обучении.